#### Aufgabe 1: Verbindung & Grundoperationen (String)

In [1]:
import redis

r = redis.Redis(host='localhost', port=6379, db=0,decode_responses=True)

r.set("username", "alice")
print(r.get("username"))         # b'alice'
print(r.exists("username"))      # 1

alice
1


#### Aufgabe2: Ablaufzeit (TTL)

In [4]:
import time

r.set("session_token", "xyz123", ex=10)
print(r.ttl("session_token"))    # z.B. 9
time.sleep(11)
print(r.exists("session_token")) # 0 -> abgelaufen

10
[PUBLISHER] Gesendet: Breaking News at 16:39:30Received: Breaking News at 16:39:30



KeyboardInterrupt: 

Received: Breaking News at 16:39:35
[PUBLISHER] Gesendet: Breaking News at 16:39:35
Received:[PUBLISHER] Gesendet: Breaking News at 16:39:40
 Breaking News at 16:39:40


#### Aufgabe 3: Zähler (String-Integer)

In [4]:
r.delete("page_views")           # Reset
for _ in range(5):
    r.incr("page_views")

print(r.get("page_views"))       # '5'

5


#### Aufgabe 4: Hash – Benutzerprofil

In [5]:
r.hset("user:1001", mapping={
    "name": "Alice",
    "age": "30",
    "email": "alice@example.com"
})
print(r.hget("user:1001", "name"))   # 'Alice'
print(r.hgetall("user:1001"))        # alles als dict
r.hset("user:1001", "age", "31")


Alice
{'name': 'Alice', 'age': '30', 'email': 'alice@example.com'}


0

#### Aufgabe 5: Liste – Aufgaben-Queue

In [ ]:
r.delete("tasks")
r.rpush("tasks", "send_email", "generate_report", "backup_db")
print(r.lpop("tasks"))    # 'send_email'
print(r.lrange("tasks", 0, -1))  # Restliche Aufgaben



#### Aufgabe 6: Set – Eindeutige Besucher

In [ ]:
r.delete("unique_visitors")
r.sadd("unique_visitors", "192.168.0.1", "192.168.0.2", "192.168.0.1")
print(r.scard("unique_visitors"))   # 2
print(r.smembers("unique_visitors"))

#### Aufgabe 7: Sorted Set – Rangliste

In [ ]:
r.delete("leaderboard")
r.zadd("leaderboard", {"Alice": 50, "Bob": 70, "Carol": 60})
r.zincrby("leaderboard", 15, "Alice")

players = r.zrevrange("leaderboard", 0, -1, withscores=True)
for p, s in players:
    print(p.decode(), s)  # Alice 65.0, Bob 70.0, Carol 60.0

#### Aufgabe 8: Transaktion

In [ ]:
from datetime import datetime

pipe = r.pipeline()
pipe.incrby("coins", 10)
pipe.set("last_update", datetime.now().isoformat())
pipe.execute()   # atomar

print(r.get("coins"))
print(r.get("last_update"))


#### Aufgabe 9: Pipeline – Performance

In [ ]:
import time

r.delete("item:0")

# Ohne Pipeline
start = time.time()
for i in range(1000):
    r.set(f"item:{i}", i)
print("Ohne Pipeline:", round(time.time() - start, 2), "s")

# Mit Pipeline
pipe = r.pipeline()
start = time.time()
for i in range(1000):
    pipe.set(f"item:{i}", i)
pipe.execute()
print("Mit Pipeline:", round(time.time() - start, 2), "s")

#### Aufgabe 10: Pub/Sub – Nachrichtenübertragung

In [ ]:
import redis
import time
import threading

# Publisher 
def publisher():
    r1 = redis.Redis(host='localhost', port=6379, db=0,decode_responses=True)
    while True:
        message = "Breaking News at " + time.strftime("%H:%M:%S")
        r1.publish("news", message)
        print(f"[PUBLISHER] Gesendet: {message}")
        time.sleep(5)

def subscriber():
    r2 = redis.Redis(host='localhost', port=6379, db=0,decode_responses=True)
    p = r2.pubsub()
    p.subscribe("news")

    print("Listening for messages...")
    for msg in p.listen():
        if msg["type"] == "message":
            print("Received:", msg["data"])
                                   
#Start in unterschiedlichen Threads     
t1 = threading.Thread(target = publisher)
t2 = threading.Thread(target = subscriber)

t1.start()
t2.start()

#### Aufgabe 11: Expiring Cache Layer

In [ ]:
import time

def getOrCache(key, func, ttl=5):
    val = r.get(key)
    if val:
        return val
    result = func()
    r.set(key, result, ex=ttl)
    return result

def slowFunction():
    time.sleep(5)
    return "fresh_result"

start = time.time()
print(getOrCache("mycache", slowFunction))  # dauert ~5s
print(getOrCache("mycache", slowFunction))  # sofort
print("Gesamtdauer:", round(time.time() - start, 2), "s")


Received:[PUBLISHER] Gesendet: Breaking News at 16:42:06
 Breaking News at 16:42:06
Received:[PUBLISHER] Gesendet: Breaking News at 16:42:11
 Breaking News at 16:42:11


#### Expertenaufgabe 12: Stream – Ereignisprotokoll

In [3]:
r.delete("events")

r.xadd("events", {"type": "login", "user": "alice"})
r.xadd("events", {"type": "purchase", "user": "bob"})
r.xadd("events", {"type": "login", "user": "max"})

events = r.xrevrange("events", count=2)

for eid, data in events:
    print(eid, {k: v for k, v in data.items()})


1770565417960-0 {'type': 'login', 'user': 'max'}
1770565417959-0 {'type': 'purchase', 'user': 'bob'}


#### Expertenaufgabe 13:Mini-Rate-Limiter

In [7]:
import time

def rateLimit(user_id, limit=5, window=60):
    key = f"rate:{user_id}"
    current = r.incr(key)
    if current == 1:
        r.expire(key, window)
    ttl = r.ttl(key)
    if current > limit:
        return False, ttl
    return True, ttl

for i in range(7):
    allowed, ttl = rateLimit("user123")
    print(f"Versuch {i+1}: {'OK' if allowed else 'BLOCKED'} (Reset in {ttl}s)")
    time.sleep(1)


Versuch 1: OK (Reset in 37s)
Versuch 2: OK (Reset in 36s)
Versuch 3: OK (Reset in 35s)
Versuch 4: BLOCKED (Reset in 34s)
Versuch 5: BLOCKED (Reset in 33s)
Versuch 6: BLOCKED (Reset in 32s)
Versuch 7: BLOCKED (Reset in 31s)


#### Expertenaufgabe 14:Lua-Script – Atomare Operation

In [4]:
lua_script = """
if redis.call('exists', KEYS[1]) == 0 then
    redis.call('set', KEYS[1], 'initialized')
    return 'OK'
else
    return 'EXISTS'
end
"""

result = r.eval(lua_script, 1, "lua:test")
print(result)

OK
